# Project 1. Denoising Network

In this project, you're going to implement a neural network to denoise images, there are several parts you need to implement to make the whole pipeline complete.

1. Dataset
2. Metrics
3. Networks
4. Training
5. Additional Question

## Dataset

In this project we are going to use an image dataset of 400 grayscale 180*180 images as our dataset, use command below to download the dataset

In [ ]:
import os
import requests # Using requests for better download control and error handling

# The second Dropbox URL seems to be the intended one for ImageSet-1.zip
zip_url = 'https://www.dropbox.com/scl/fi/bhvpke5p7u1xolerayx13/ImageSet-1.zip?rlkey=1oz9g3cpomt0wln0es8j9wje6&st=1fj2ksrj&dl=1'
zip_filename = 'ImageSet.zip'

print(f"Attempting to download from: {zip_url}")

try:
    # Use requests for a more robust download, following redirects
    response = requests.get(zip_url, stream=True)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(zip_filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded {zip_filename}.")

    # Check if the downloaded file looks like a valid zip by its size
    file_size = os.path.getsize(zip_filename)
    # A 400-image dataset (180x180 grayscale) compressed should be at least a few MBs.
    # 1MB is a conservative lower bound for a valid zip.
    if file_size < 1024 * 1024:
        print(f"Warning: Downloaded file '{zip_filename}' is suspiciously small ({file_size / (1024*1024):.2f} MB). It might not be a valid zip archive.")
        print("Please check the Dropbox link or try downloading manually.")
        # Remove the invalid file to avoid further errors
        os.remove(zip_filename)
        raise ValueError("Invalid zip file downloaded due to small size.")

    # Unzip the file silently
    !unzip -q {zip_filename}

    # Rename ImageSet-1 to ImageSet if the alternate URL was used and created that folder
    # This part remains from the original logic, assuming unzip might create ImageSet-1
    if not os.path.exists('ImageSet') and os.path.exists('ImageSet-1'):
        os.rename('ImageSet-1', 'ImageSet')
    # Also ensure the ImageSet folder exists if the zip directly created it
    elif not os.path.exists('ImageSet') and os.path.exists('ImageSet-1'):
        os.rename('ImageSet-1', 'ImageSet') # Redundant check but ensures consistency

    # Verify that the 'ImageSet' directory now exists
    if os.path.exists('ImageSet'):
        print('Download and extraction complete.')
        print('Files in ImageSet:', len(os.listdir('ImageSet')))
    else:
        print("Error: 'ImageSet' directory not found after extraction. The zip might have a different internal structure.")
        print("Please check the contents of the zip file.")

except requests.exceptions.RequestException as e:
    print(f"Error downloading file: {e}")
    print("Please check your internet connection or the provided Dropbox link.")
except ValueError as e:
    print(f"Error during file processing: {e}")
except Exception as e:
    print(f"An unexpected error occurred during download or extraction: {e}")
    print("Failed to set up ImageSet. Please verify the source.")


If above link does not work, please use:
https://www.dropbox.com/scl/fi/bhvpke5p7u1xolerayx13/ImageSet-1.zip?rlkey=1oz9g3cpomt0wln0es8j9wje6&st=1fj2ksrj&dl=0
Now you should have a folder called ImageSet, and there're 400 images in it

In [ ]:
!ls ImageSet | wc -l

Now you need to implement two classes, TrainingSet and TestingSet, you should first split your dataset into 350 images and 50 images. TrainingSet should use the 350 images to form a dataset, with each entry being a pair of image tensors, and the first image should be a noisy version of the second original image. In other words, `training_set[i]` should return `[noisy_image(=original_image + noise), original_image]`, and images should be tensors of shape $C\times H\times W$, in this case, $1\times 180\times 180$

TestingSet is the same thing with the remaining 50 images.
1. Please refer to the following code to add noise
    ```python
    def add_noise(img):
        noise = torch.randn(img.size()).mul_(self.sigma/255.0)
        noisy = img + noise
        noisy[torch.where(noisy > 1)] = 1
        noisy[torch.where(noisy < 0)] = 0
        return noisy
    ```
2. Also refer to the following code as how to read images from file to memory
    ```python
    image_path = os.path.join(ROOT_PATH, IMAGE_PATH)
    img = PIL.Image.open(image_path)
    ```

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from PIL import Image


class TrainingSet(Dataset):
    def __init__(self, root='ImageSet', sigma=25):
        self.sigma = sigma
        self.image_paths = sorted([
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
        ])[:350]  # first 350 images for training
        self.to_tensor = transforms.ToTensor()

    def add_noise(self, img):
        noise = torch.randn(img.size()).mul_(self.sigma / 255.0)
        noisy = img + noise
        noisy.clamp_(0, 1)
        return noisy

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        img_tensor = self.to_tensor(img)  # [1, H, W]
        noisy_tensor = self.add_noise(img_tensor.clone())
        return noisy_tensor, img_tensor


class TestingSet(Dataset):
    def __init__(self, root='ImageSet', sigma=25):
        self.sigma = sigma
        self.image_paths = sorted([
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
        ])[350:]  # last 50 images for testing
        self.to_tensor = transforms.ToTensor()

    def add_noise(self, img):
        noise = torch.randn(img.size()).mul_(self.sigma / 255.0)
        noisy = img + noise
        noisy.clamp_(0, 1)
        return noisy

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        img_tensor = self.to_tensor(img)  # [1, H, W]
        noisy_tensor = self.add_noise(img_tensor.clone())
        return noisy_tensor, img_tensor


You can use the following code block to check if your implementation is correct, first, there should be **no error**, second, the shape of image should be **`[1, 180, 180]`**, and finally, in the drawing area, the **left hand side image should be noisier than the right hand side image**, but they should be images of the same thing.

In [ ]:
training_set = TrainingSet()
testing_set = TestingSet()
assert len(training_set) == 350
assert len(testing_set) == 50

print(f'Shape of image: {training_set[0][0].shape}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2)
axes[0].imshow(training_set[0][0][0], cmap='gray')
axes[0].axis('off')
axes[0].set_title('noisy example')
axes[1].imshow(training_set[0][1][0], cmap='gray')
axes[1].axis('off')
axes[1].set_title('original example')

## Metrics
To quantify how noisy an image is compared to the original one, we're going to use PSNR, please implement a function `psnr` to return the psnr score.

Refer to https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio about the formula of PSNR

Note:
1. higher PSNR means noise is relatively smaller, the PSNR of the original image is positive infinity, because the noise is zero.
2. the psnr is a symetric function, meaning the psnr of a noisy image with respect to the original one is the same as the psnr of the original image with respect to the noisy one.

In [ ]:
import torch

def psnr(original, noisy):
    """PSNR between two image tensors with pixel values in [0, 1]."""
    mse = torch.mean((original.float() - noisy.float()) ** 2)
    if mse == 0:
        return torch.tensor(float('inf'), device=mse.device)
    return 10.0 * torch.log10(torch.tensor(1.0, device=mse.device) / mse)


Run the following code to check if the implementation is correct, the expected output should be about 7.96

In [ ]:
import torch
test_original = torch.tensor([[0.1, 0.2], [0.3, 0.4]])
test_noisy = torch.tensor([[0.5, 0.6], [0.7, 0.8]])
print(f'PSNR score: {psnr(test_original, test_noisy)}')

And we can calculate the psnr score for the noisy image pair we showed above, the score should be aroud 28, but there could be exception.

In [ ]:
print(f'PSNR score for example images: {psnr(training_set[0][1], training_set[0][0])}')

## Network
Now that we got dataset ready and metrics ready, we start preparing the network. You need to define a class `DenoiseNetwork` as your network class.

The goal of your network is to take the noisy image as input and output the predicted **noise**. First of all, the input and the output of the network should have the same size, the main idea is to predict the original image first by going through several CNN layers, and then use the input noisy image to deduct predicted original image to get the noise, the pseudo code should be like:
```python
class DenoiseNetwork(nn.Module):
    def __init__(self):
        define some cnn layers and other necessary components
    
    def forward(self, x):
        predicted_original_image = cnn_network(x)
        noise = x - predicted_original_image
        return noise
```
Then calculate the mean squared error between the predicted noise and the truth noise as our loss, and try to minimize it.

Tips:
1. you can use nn.MSELoss as your loss function
2. Use Adam instead of SGD as your optimizer, initial learning rate set to 0.001
3. Use `torch.nn.init.orthogonal_` to initialize the `weight` of your cnn layers as orthogonal matrices, and use `torch.nn.init.constant_` to fill the `bias` of your cnn layers with `0`s.
4. Try dropout, batchnorm etc. to improve the results (training speed, restored results etc.)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init

# Detect GPU; fall back to CPU if no GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


class DenoiseNetwork(nn.Module):
    """
    DnCNN-inspired residual denoising network.
    Predicts the noise; denoised = noisy - predicted_noise.
    Architecture: Conv+ReLU -> (Conv+BN+ReLU) x (num_layers-2) -> Conv
    """
    def __init__(self, num_layers=7, num_channels=64):
        super(DenoiseNetwork, self).__init__()
        layers = []

        # First layer: Conv + ReLU (no BN)
        layers.append(nn.Conv2d(1, num_channels, kernel_size=3, padding=1, bias=True))
        layers.append(nn.ReLU(inplace=True))

        # Middle layers: Conv + BN + ReLU
        for _ in range(num_layers - 2):
            layers.append(nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1, bias=True))
            layers.append(nn.BatchNorm2d(num_channels))
            layers.append(nn.ReLU(inplace=True))

        # Last layer: Conv only (no BN, no activation)
        layers.append(nn.Conv2d(num_channels, 1, kernel_size=3, padding=1, bias=True))

        self.cnn = nn.Sequential(*layers)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                init.orthogonal_(m.weight)
                init.constant_(m.bias, 0)

    def forward(self, x):
        predicted_original = self.cnn(x)
        noise = x - predicted_original
        return noise

    @property
    def device(self):
        return next(self.parameters()).device


net = DenoiseNetwork().to(device)


Here're some basic tests to see if your network can at least run through an example image, this is expected to produce no error.

In [ ]:
example_batch = training_set[0][0].unsqueeze(0)
assert net(example_batch.to(device)).shape == example_batch.shape
print('Network shape test passed.')


Now we need a quantitative score to indicate how well a network performs. Previously we have defined the psnr function, but it only calculates psnr of an image pair, we need to calculate two scores to see how well the network denoises, the first is the mean psnr score of all noisy images, which indicates how noisy these unprocessed images are, and then assume we have the network ready, we can use the network to predict the noise, and deduct the noise from the noisy images to produce restored images, then we calculate the mean psnr score of these restored images with respect to the original images, and this score indicate how noisy the restored images are. If everything works out fine, we should be able to observe a higher psnr on the restored images.

You need to define a `mean_psnr` function that takes a dataset and a network as input and calculate the mean psnr scores of original noisy images across the whole dataset and mean psnr of restored images processed by the network.

In [ ]:
def mean_psnr(testset, net):
    """
    Returns (mean_psnr_original, mean_psnr_after):
      mean_psnr_original : mean PSNR of noisy images vs. originals
      mean_psnr_after    : mean PSNR of network-restored images vs. originals
    """
    net.eval()
    psnr_original_list = []
    psnr_after_list = []

    with torch.no_grad():
        for i in range(len(testset)):
            noisy_img, original_img = testset[i]
            noisy_img = noisy_img.to(net.device)
            original_img = original_img.to(net.device)

            psnr_original_list.append(psnr(original_img, noisy_img))

            predicted_noise = net(noisy_img.unsqueeze(0)).squeeze(0)
            restored_img = (noisy_img - predicted_noise).clamp(0, 1)

            psnr_after_list.append(psnr(original_img, restored_img))

    mean_psnr_original = torch.stack(psnr_original_list).mean()
    mean_psnr_after = torch.stack(psnr_after_list).mean()
    return mean_psnr_original, mean_psnr_after


We can calculte the mean psnr on `testing_set`

In [ ]:
mean_psnr(testing_set, net)

If your code is correct, you should see the mean psnr of original images should be around 28, and the psnr of network processed images is much smaller, which means, a randomly initialzed network adds even more noise, you should see this by displaying.

In [ ]:
import matplotlib.pyplot as plt

noisy_image, original_image = testing_set[0]
noisy_image = noisy_image.to(device)
with torch.no_grad():
    predicted_noise = net(noisy_image.unsqueeze(0)).squeeze(0)
restored_image = (noisy_image - predicted_noise).clamp(0, 1)

fig, axes = plt.subplots(1, 3)
fig.set_figwidth(15)
axes[0].imshow(noisy_image[0].cpu(), cmap='gray')
axes[0].axis('off')
axes[0].set_title('noisy')
axes[1].imshow(original_image[0], cmap='gray')
axes[1].axis('off')
axes[1].set_title('original')
axes[2].imshow(restored_image[0].cpu().detach(), cmap='gray')
axes[2].axis('off')
axes[2].set_title('restored (before training)')
plt.show()


## Training
Now that we got everything ready, we should start training, in the next section, you need to implement the training process, that includes defining criteria, setting up optimizer, going through several epochs to train the network, during the training, you should also analyze the psnr scores to see how it goes in terms of quantified performance.

Checklist:
1. define dataloader, recommend batch size starting from 32
2. criteria
3. optimizer
4. (optional) consider using functions in torch.optim.lr_scheduler to adjust your learning rate, because smaller learning rate might work better in the later period of training, similar to fine adjustment. Reference: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
5. during each iteration, you need to 1. get the noisy image and the original image 2. calculate predicted noise from network, use MSE to calculate the distance between predicted noise and true noise 3. reset gradients to zero 3. use the distance as loss to backward the network to get gradients 4. perform learning with the gradients using optimizer
6. From time to time (e.g. each epoch), calculate PSNR on testing_set

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 0.001

# num_workers=0 avoids multiprocessing issues inside Colab/Jupyter
train_loader = DataLoader(training_set, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)

criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)

# Reduce LR at epoch 10 and 20 for finer convergence
scheduler = optim.lr_scheduler.MultiStepLR(optimizer,
                                            milestones=[10, 20], gamma=0.1)

print('Starting training...')
print(f"{'Epoch':>6} | {'Train Loss':>12} | {'PSNR Noisy':>12} | {'PSNR Restored':>14}")
print('-' * 52)

for epoch in range(1, NUM_EPOCHS + 1):
    net.train()
    total_loss = 0.0

    for noisy_imgs, original_imgs in train_loader:
        noisy_imgs = noisy_imgs.to(device)
        original_imgs = original_imgs.to(device)

        # true noise = difference between noisy and clean (after clamping)
        true_noise = noisy_imgs - original_imgs
        predicted_noise = net(noisy_imgs)
        loss = criterion(predicted_noise, true_noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(train_loader)
    psnr_before, psnr_after = mean_psnr(testing_set, net)
    print(f'{epoch:>6} | {avg_loss:>12.6f} | '
          f'{psnr_before.item():>12.4f} | {psnr_after.item():>14.4f}')

print('Training complete!')


Now that your net is ready, we can re do the demonstration.

In [ ]:
noisy_image, original_image = testing_set[0]
noisy_image = noisy_image.to(device)
with torch.no_grad():
    predicted_noise = net(noisy_image.unsqueeze(0)).squeeze(0)
restored_image = (noisy_image - predicted_noise).clamp(0, 1)

fig, axes = plt.subplots(1, 3)
fig.set_figwidth(15)
axes[0].imshow(noisy_image[0].cpu(), cmap='gray')
axes[0].axis('off')
axes[0].set_title('noisy')
axes[1].imshow(original_image[0], cmap='gray')
axes[1].axis('off')
axes[1].set_title('original')
axes[2].imshow(restored_image[0].cpu().detach(), cmap='gray')
axes[2].axis('off')
axes[2].set_title('restored (after training)')
plt.show()


The network I trained here is a simple 3-layer low number of channel cnn network, and you can see it's already starting to work. Now try adjust some parameters/network structure to make it work even better. Write down your analysis to make a pdf report.

You need to submit two files, this ipynb file and a pdf report with your analysis.

# Additional Question

In this additional question, you need to test on noise different from your training set. In the previous project, you were asked to use Gaussian noise for training, and you also used Gaussian noise when testing.

In this additional question, you are asked to test your trained model using speckle noise. A simple model of Speckle noise is multiplicative noise, which means that the noise is generated by multiplying each pixel value of the image by a random number. This random number is usually drawn from a distribution with a mean of 1 to ensure that noise averaging does not brighten or darken the image.

Below is a Python function that simply implements Speckle noise. This implementation first generates a random noise matrix of the same size as the input image, with values drawn from a normal distribution with mean 1 and standard deviation sigma. This noise matrix is then multiplied by the original image to generate an image with Speckle noise. Finally, the result is constrained to the range 0 to 1 to keep the pixel values valid.

```python
def add_speckle_noise(img, sigma=0.1):
    noise = torch.randn(img.size()) * sigma + 1.0
    noisy_img = img * noise
    noisy_img.clamp_(0, 1)
    return noisy_img
```
You are asked to do the following:

1. First, you are asked to find appropriate sigma values so that the resulting image is visually similar to your training noise. To make your results more general, it is best to choose 3 possible values. (5%)

2. Test your model on these speckle noises using the same model trained above. Compare the PSNR tested on speckle noise with the PSNR tested on Gaussian noise. Show your results in a table. (5%)

3. Finally, draw your conclusion in the pdf report. Can a model trained on Gaussian noise handle speckle noise? To what extent can it be handled? (5%)


In [ ]:
import torch
import matplotlib.pyplot as plt


def add_speckle_noise(img, sigma=0.1):
    """Multiplicative speckle noise: pixel *= N(1, sigma)."""
    noise = torch.randn(img.size(), device=img.device) * sigma + 1.0
    noisy_img = img * noise
    noisy_img.clamp_(0, 1)
    return noisy_img


# --- Part 1: Find sigma values with similar noise level to Gaussian sigma=25 ---
# Gaussian sigma=25 gives PSNR ~28 dB; scan speckle sigmas to match.
speckle_sigmas = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]

print('Speckle sigma vs. mean PSNR (noisy):')
print(f"{'Sigma':>8} | {'Mean PSNR':>10}")
print('-' * 22)

sigma_psnr = {}
for sig in speckle_sigmas:
    psnr_vals = []
    for i in range(len(testing_set)):
        _, orig = testing_set[i]
        noisy = add_speckle_noise(orig.clone(), sigma=sig)
        psnr_vals.append(psnr(orig, noisy))
    mean_p = torch.stack(psnr_vals).mean().item()
    sigma_psnr[sig] = mean_p
    print(f'{sig:>8.2f} | {mean_p:>10.4f}')

# Three sigmas nearest to Gaussian training noise (~28 dB)
chosen_sigmas = [0.08, 0.10, 0.12]
print(f'\nChosen sigma values (visually similar to Gaussian sigma=25): {chosen_sigmas}')


# --- Part 2: PSNR comparison table ---
net.eval()

print('\n--- PSNR Comparison Table ---')
print(f"{'Noise Type':>20} | {'Sigma':>8} | {'PSNR Noisy':>12} | {'PSNR Restored':>14}")
print('-' * 62)

# Gaussian (training distribution)
gauss_before, gauss_after = mean_psnr(testing_set, net)
print(f"{'Gaussian':>20} | {'25':>8} | "
      f"{gauss_before.item():>12.4f} | {gauss_after.item():>14.4f}")

# Speckle noise at each chosen sigma
for sig in chosen_sigmas:
    psnr_noisy_list = []
    psnr_restored_list = []
    with torch.no_grad():
        for i in range(len(testing_set)):
            _, orig = testing_set[i]
            noisy = add_speckle_noise(orig.clone(), sigma=sig)
            noisy_dev = noisy.to(net.device)
            orig_dev = orig.to(net.device)

            psnr_noisy_list.append(psnr(orig_dev, noisy_dev))

            pred_noise = net(noisy_dev.unsqueeze(0)).squeeze(0)
            restored = (noisy_dev - pred_noise).clamp(0, 1)
            psnr_restored_list.append(psnr(orig_dev, restored))

    mean_noisy = torch.stack(psnr_noisy_list).mean().item()
    mean_restored = torch.stack(psnr_restored_list).mean().item()
    print(f"{'Speckle':>20} | {sig:>8.2f} | "
          f"{mean_noisy:>12.4f} | {mean_restored:>14.4f}")


# --- Visualization: Gaussian vs Speckle ---
_, orig_img = testing_set[0]
gauss_noisy_img, _ = testing_set[0]
speckle_noisy_img = add_speckle_noise(orig_img.clone(), sigma=0.10)

with torch.no_grad():
    gauss_pred = net(gauss_noisy_img.unsqueeze(0).to(net.device)).squeeze(0)
    gauss_restored = (gauss_noisy_img.to(net.device) - gauss_pred).clamp(0, 1)
    speckle_pred = net(speckle_noisy_img.unsqueeze(0).to(net.device)).squeeze(0)
    speckle_restored = (speckle_noisy_img.to(net.device) - speckle_pred).clamp(0, 1)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles = [['Gaussian Noisy', 'Original', 'Gaussian Restored'],
          ['Speckle Noisy (sigma=0.10)', 'Original', 'Speckle Restored']]
# Move all tensors to CPU before converting to numpy for plotting
imgs = [
    [gauss_noisy_img[0].cpu(), orig_img[0].cpu(), gauss_restored[0].cpu().detach()],
    [speckle_noisy_img[0].cpu(), orig_img[0].cpu(), speckle_restored[0].cpu().detach()],
]

for r in range(2):
    for c in range(3):
        axes[r][c].imshow(imgs[r][c].numpy(), cmap='gray')
        axes[r][c].axis('off')
        axes[r][c].set_title(titles[r][c])
plt.tight_layout()
plt.savefig('speckle_vs_gaussian.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualization saved to speckle_vs_gaussian.png')


# Marking Scheme:


*   Code implementation: 35%


> * Dataset 5% (Download the dataset correctly and preprocess it appropriately for use as the training and testing sets.)
> * Metrics 5% (Set and use appropriate metrics to evaluate the model.)
> * Network 5% (only 5% because network overlaps with results, you need to adjust the network to improve the results anyway.)
> * Training code 10% (The training code executes correctly, and the code structure is clear and well organized.)
> * reasonably good results 10% (The model reaches the target training outcome with satisfactory performance.)


*   PDF report: 25%

> * Basic results demonstration (network introduction, denoising results showcase) 10%
> * Analysis and improvements 15% (You're supposed to clarify how do you make the network work, e.g. if you encounter some issues, what do you do to address them)


*   Addtional question: 15%

> * Find appropriate sigma values (5%)
> * Test your model on these speckle noises and report them in a table (5%)
> * Draw your conclusion: Can a model trained on Gaussian noise handle speckle noise? To what extent can it be handled? (5%)


*   Oral question: 25%

> * After the assignment submission, each group will be asked one question related to the assignment during the tutorial. This is a group-based activity, and each group may nominate one member to answer the question. All group members must attend the session.

